# 03 Factor Screening

Generate a multi-factor ranking table, inspect coverage and stability fields, and review the qualified-factor report.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def locate_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "apps").exists():
            return candidate
    raise RuntimeError("repo root not found")


REPO_ROOT = locate_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from apps.quant_platform.research.data_loader import ResearchDataLoader
from apps.quant_platform.research.scripts.run_factor_research import run_factor_research

RESEARCH_ROOT = REPO_ROOT / "apps/quant_platform/research"
OUTPUT_ROOT = RESEARCH_ROOT / "output/notebook_screening"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
loader = ResearchDataLoader()

In [ ]:
panel = loader.prepare_panel(
    loader.load_panel(
        start_date="2024-01-01",
        end_date="2024-06-30",
        columns=[
            "ts_code", "trade_date", "open", "close", "pct_chg", "turnover_rate_f", "volume_ratio",
            "pe_ttm", "pb", "ps_ttm", "dv_ttm",
        ],
    )
)
factor_cols = ["pct_chg", "turnover_rate_f", "volume_ratio", "pe_ttm", "pb", "ps_ttm", "dv_ttm"]
panel.head()

In [ ]:
screening = run_factor_research(
    panel,
    factor_cols=factor_cols,
    target_col="overnight_return",
    output_dir=OUTPUT_ROOT,
)
ranking = screening["ranking"]
ranking[[
    "factor_name", "mean_ic", "rank_ic", "ic_ir", "coverage",
    "active_trade_date_coverage", "rolling_1y_valid_ratio", "long_short_mean",
]].head(20)

In [ ]:
qualified = ranking.loc[
    ranking["coverage"].ge(0.3)
    & ranking["rolling_1y_valid_ratio"].ge(0.0)
    & ranking["ic_ir"].abs().ge(0.1)
]
qualified.to_csv(OUTPUT_ROOT / "03_qualified_factors.csv", index=False)
display(qualified.head(20))
Path(screening["ranking_overview_html"]), Path(screening["correlation_plot"])

## Follow-up

- Review `factor_ranking.html` to scan top factors quickly.
- Promote only factors with acceptable coverage and stable IC sign into composite construction.
- If many valuation factors survive while event factors do not, revisit event lag handling and coverage thresholds.